# Daty i godziny w pandas — kompletny przewodnik

**Problem:** operacje na datach w pandas są rozproszone między kilka mechanizmów (`.dt` accessor, `Grouper`, `resample`, `asfreq`, `Period`), które łatwo pomylić — zwłaszcza że kilka z nich wygląda podobnie, ale działa inaczej (`.dt.date` vs `.dt.normalize()`, `Grouper` vs `resample`, `asfreq` vs `resample`).

**Zakres notatki:** ekstrakcja komponentów daty, konwersja data↔tekst, okresy (`Period`), przesunięcia w czasie, grupowanie i agregacja po datach, oraz uzupełnianie brakujących dat przy datach jako indeksie.

**Jak korzystać:** każda sekcja jest samodzielna — możesz wracać bezpośrednio do interesującego Cię fragmentu. Pełna tabela z mapowaniem "zadanie → rozwiązanie" jest na końcu.

## Setup

In [ ]:
import pandas as pd
import numpy as np

# Dane transakcyjne z pełnym znacznikiem czasu (data + godzina), kilka transakcji w tym samym dniu
transactions = pd.DataFrame({
    "transaction_time": pd.to_datetime([
        "2026-01-03 08:15:00", "2026-01-03 14:42:00", "2026-01-05 09:03:00",
        "2026-02-01 11:20:00", "2026-02-14 16:55:00", "2026-02-20 10:10:00",
        "2026-03-02 07:45:00", "2026-03-15 13:30:00",
    ]),
    "amount": [120.50, 89.90, 340.00, 210.25, 75.60, 430.10, 198.00, 302.75],
})

transactions

## Sekcja 1 — Konwersja tekstu na datę

Zanim zadziała accessor `.dt`, kolumna musi mieć typ `datetime64`, nie `object`/`string`. `errors="coerce"` zamienia niesparsowalne wartości na `NaT` zamiast przerywać całą konwersję wyjątkiem — kluczowe przy danych z realnych eksportów, gdzie zawsze trafi się kilka błędnych wpisów.

In [ ]:
print(transactions["transaction_time"].dtype)  # już datetime64[ns] - w setupie użyto pd.to_datetime

# Typowa sytuacja: kolumna z tekstem, część wpisów błędna
messy_dates = pd.Series(["2026-01-01", "not a date", "2026-01-03", ""])
parsed = pd.to_datetime(messy_dates, errors="coerce")
print(parsed)
print(f"\nLiczba błędnych/pustych wartości (NaT): {parsed.isna().sum()}")

## Sekcja 2 — Wyciąganie komponentów daty (`.dt`)

Wszystkie poniższe działają tylko na kolumnie typu `datetime64` (accessor `.dt`). Uwaga: `.dt.dayofweek` liczy od **0 = poniedziałek** — inaczej niż w Excelu, gdzie typowo 1 = niedziela. To częste źródło przesunięcia o jeden dzień przy migracji logiki z Excela.

In [ ]:
transactions["year"] = transactions["transaction_time"].dt.year
transactions["month"] = transactions["transaction_time"].dt.month
transactions["month_name"] = transactions["transaction_time"].dt.month_name()
transactions["day"] = transactions["transaction_time"].dt.day
transactions["hour"] = transactions["transaction_time"].dt.hour
transactions["day_of_week"] = transactions["transaction_time"].dt.dayofweek  # 0 = poniedziałek
transactions["day_name"] = transactions["transaction_time"].dt.day_name()
transactions["quarter"] = transactions["transaction_time"].dt.quarter
transactions["iso_week"] = transactions["transaction_time"].dt.isocalendar().week

transactions[[
    "transaction_time", "year", "month", "month_name",
    "day", "hour", "day_of_week", "day_name", "quarter", "iso_week",
]]

## Sekcja 3 — Kolumna z samą datą, bez godziny

Dwa sposoby, które dają wizualnie ten sam wynik, ale **różny typ danych** — a to ma realne konsekwencje dla dalszych operacji.

- `.dt.date` → zwraca obiekt Python `datetime.date`, typ kolumny to `object`. Traci się możliwość dalszych operacji `.dt` i arytmetyki na `Timestamp` (np. dodawania `pd.Timedelta`).
- `.dt.normalize()` → zeruje godzinę do `00:00:00`, ale **zostaje** `datetime64[ns]`. Można dalej używać `.dt`, `resample()`, dodawania offsetów itd.

**Zasada:** używaj `.dt.normalize()`, jeśli data będzie dalej przetwarzana w pandas. `.dt.date` zostaw na sam koniec — do wyświetlenia lub eksportu (np. do Excela/CSV).

In [ ]:
transactions["date_as_object"] = transactions["transaction_time"].dt.date
transactions["date_normalized"] = transactions["transaction_time"].dt.normalize()

print(f"dt.date dtype:       {transactions['date_as_object'].dtype}")        # object
print(f"dt.normalize dtype:  {transactions['date_normalized'].dtype}")        # datetime64[ns]

transactions[["transaction_time", "date_as_object", "date_normalized"]]

## Sekcja 4 — Konwersja daty na tekst (`.dt.strftime`)

Najczęściej używane kody formatowania:

| Kod | Znaczenie | Przykład |
|---|---|---|
| `%Y` | rok (4 cyfry) | 2026 |
| `%y` | rok (2 cyfry) | 26 |
| `%m` | miesiąc (01–12) | 03 |
| `%B` | pełna nazwa miesiąca | March |
| `%b` | skrócona nazwa miesiąca | Mar |
| `%d` | dzień miesiąca | 15 |
| `%A` | pełna nazwa dnia tygodnia | Sunday |
| `%a` | skrócona nazwa dnia tygodnia | Sun |
| `%H` | godzina, 24h (00–23) | 14 |
| `%M` | minuta | 42 |
| `%S` | sekunda | 05 |

In [ ]:
transactions["date_str_pl"] = transactions["transaction_time"].dt.strftime("%d.%m.%Y")
transactions["date_str_iso"] = transactions["transaction_time"].dt.strftime("%Y-%m-%d")
transactions["datetime_str_full"] = transactions["transaction_time"].dt.strftime("%d.%m.%Y %H:%M")

transactions[["transaction_time", "date_str_pl", "date_str_iso", "datetime_str_full"]]

## Sekcja 5 — Okresy (`.dt.to_period`)

`Period` reprezentuje **cały przedział czasu** (np. "marzec 2026"), nie punkt w czasie jak `Timestamp`. Przydatne jako etykieta do grupowania/raportowania, gdy konkretny dzień w miesiącu jest nieistotny — dwie transakcje z różnych dni tego samego miesiąca dostaną identyczny `Period`, co ułatwia późniejsze `groupby()`.

In [ ]:
transactions["period_month"] = transactions["transaction_time"].dt.to_period("M")
transactions["period_quarter"] = transactions["transaction_time"].dt.to_period("Q")

print(transactions["period_month"].dtype)  # period[M]
transactions[["transaction_time", "period_month", "period_quarter"]]

## Sekcja 6 — Przesunięcia w czasie

Dwa różne rodzaje "przesunięcia" — łatwo je pomylić:

- **`.shift()` / `.diff()`** — przesuwają **wartości** względem kolejności wierszy (np. "sprzedaż z poprzedniego dnia"). Nie zmieniają samej kolumny z datą.
- **`+ pd.DateOffset(...)` / `+ pd.Timedelta(...)`** — przesuwają **samą datę** o konkretny odcinek czasu. `DateOffset` rozumie kalendarz (np. "miesiąc" ma różną liczbę dni), `Timedelta` liczy w sztywnych jednostkach (dni/godziny/minuty).

In [ ]:
daily_sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=5, freq="D"),
    "sales": [100, 120, 90, 150, 130],
})

# Przesunięcie WARTOŚCI - o ile zmieniła się sprzedaż względem poprzedniego wiersza
daily_sales["sales_previous_day"] = daily_sales["sales"].shift(1)
daily_sales["day_over_day_change"] = daily_sales["sales"].diff()

# Przesunięcie samej DATY o kalendarzowy odcinek czasu
daily_sales["date_plus_1_month"] = daily_sales["date"] + pd.DateOffset(months=1)
daily_sales["date_plus_3_days"] = daily_sales["date"] + pd.Timedelta(days=3)

daily_sales

## Sekcja 7 — Grupowanie po dacie: `pd.Grouper`

`pd.Grouper(key="kolumna", freq="...")` grupuje po dacie **bez** ustawiania jej jako indeks — kolumna z datą zostaje zwykłą kolumną. Można łączyć z innymi kluczami grupowania w jednym `groupby()`, np. `groupby([pd.Grouper(...), "kategoria"])`.

Najczęstsze aliasy `freq`:

| Alias | Znaczenie |
|---|---|
| `D` | dzień kalendarzowy |
| `B` | dzień roboczy |
| `W` | tydzień |
| `ME` | koniec miesiąca (w starszych wersjach pandas: `M` — przestarzałe od pandas 2.2) |
| `MS` | początek miesiąca |
| `QE` | koniec kwartału |
| `YE` | koniec roku |
| `h` | godzina |
| `min` | minuta |

In [ ]:
monthly_totals = transactions.groupby(pd.Grouper(key="transaction_time", freq="ME"))["amount"].sum()
monthly_totals

## Sekcja 8 — Agregacja po dacie: `.resample()`

`.resample()` robi to samo co `Grouper`, ale **wymaga `DatetimeIndex`** — datę trzeba najpierw ustawić jako indeks (`set_index()`). W zamian daje dostęp do dodatkowych metod czasowych po agregacji (np. `.ffill()`, `.interpolate()`), które przy zwykłym `groupby()` nie są dostępne.

In [ ]:
transactions_indexed = transactions.set_index("transaction_time")

monthly_resampled = transactions_indexed["amount"].resample("ME").sum()
print(monthly_resampled)

# Kilka agregacji naraz
monthly_stats = transactions_indexed["amount"].resample("ME").agg(["sum", "mean", "count"])
monthly_stats

## Sekcja 9 — Daty jako indeks i uzupełnianie brakujących dat

Typowa sytuacja: dane mają luki (np. dni bez żadnej transakcji w ogóle nie pojawiają się w źródle), a do dalszej analizy (wykresy czasowe, obliczenia rolling) potrzebny jest pełny, regularny zakres dat.

In [ ]:
# Dane z lukami - brak wpisów za część dni (2026-01-03, 04, 06, 08, 09 nie istnieją w danych)
sparse_data = pd.DataFrame({
    "date": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-05", "2026-01-07", "2026-01-10"]),
    "visits": [50, 45, 60, 55, 70],
}).set_index("date")

sparse_data

### `asfreq()` — wymuszenie regularnej częstotliwości

`asfreq("D")` wstawia brakujące daty i wypełnia je `NaN` (bez żadnej agregacji — to nie jest to samo co `resample`, patrz Sekcja 10). Sposób wypełnienia braków zależy od charakteru danych.

In [ ]:
full_range = sparse_data.asfreq("D")
full_range  # brakujące dni: NaN

# Strategie uzupełnienia - dobór zależy od znaczenia danych:
filled_zero = sparse_data.asfreq("D", fill_value=0)     # np. liczba odwiedzin - brak dnia = 0
filled_forward = sparse_data.asfreq("D").ffill()          # np. stan magazynowy - ostatnia znana wartość się utrzymuje
filled_interpolated = sparse_data.asfreq("D").interpolate()  # np. wartość ciągła - interpolacja liniowa między punktami

filled_zero

### `reindex()` — alternatywa z jawnie zdefiniowanym zakresem

Przydatne, gdy potrzebny zakres dat nie pokrywa się dokładnie z pierwszą/ostatnią datą w danych (np. pełny miesiąc kalendarzowy, nawet jeśli dane zaczynają się później).

In [ ]:
full_date_range = pd.date_range(start="2026-01-01", end="2026-01-15", freq="D")
reindexed = sparse_data.reindex(full_date_range, fill_value=0)
reindexed

## Sekcja 10 — `asfreq()` vs `resample()`: pułapka cichej utraty danych

- **`asfreq()`** buduje nową, regularną siatkę dat zaczynając od **pierwszego** znacznika czasu w indeksie (razem z jego godziną!) i zachowuje tylko te oryginalne wiersze, które trafiają dokładnie w punkty tej siatki. Nie agreguje i nie ostrzega — wiersze, które się nie trafiają, po prostu znikają (`NaN`), a ich wartości są tracone bez śladu.
- **`resample()`** agreguje wszystkie wiersze do właściwego przedziału niezależnie od dokładnej godziny, więc żadne dane nie giną.

**To nie jest różnica teoretyczna.** Poniżej: te same dane transakcyjne, ta sama kolumna. `asfreq("D")` zachowuje tylko wiersze, których godzina zgadza się z pierwszym wpisem (2026-01-03 08:15) — czyli w praktyce tylko 1 z 8 transakcji. `resample("D").sum()` poprawnie sumuje wszystko.

**Zasada:** nigdy nie używaj `asfreq()` bezpośrednio na danych z nieregularnymi znacznikami czasu w ciągu dnia. Najpierw zagreguj (`resample()` albo `.dt.normalize()` + `groupby`) do jednoznacznych wartości dziennych, dopiero potem ewentualnie `asfreq()` do uzupełnienia luk.

In [ ]:
result_asfreq = transactions_indexed["amount"].asfreq("D")

print(f"Suma oryginalna:            {transactions['amount'].sum():.2f}")
print(f"Suma po asfreq (bez NaN):   {result_asfreq.dropna().sum():.2f}")
print(f"Zachowane wiersze:          {result_asfreq.notna().sum()} z {len(transactions)} oryginalnych\n")

# resample agreguje poprawnie - żadna transakcja nie ginie
result_resample = transactions_indexed["amount"].resample("D").sum()
print(f"Suma po resample:           {result_resample.sum():.2f}")

## Podsumowanie — mapa "zadanie → rozwiązanie"

| Zadanie | Rozwiązanie |
|---|---|
| Konwersja tekstu na datę (z tolerancją na błędy) | `pd.to_datetime(col, errors="coerce")` |
| Wyciągnięcie roku / miesiąca / dnia / godziny | `.dt.year` / `.dt.month` / `.dt.day` / `.dt.hour` |
| Nazwa miesiąca / dnia tygodnia | `.dt.month_name()` / `.dt.day_name()` |
| Numer dnia tygodnia (0 = poniedziałek) | `.dt.dayofweek` |
| Numer tygodnia w roku (ISO) | `.dt.isocalendar().week` |
| Data bez godziny, zostaje `datetime64` | `.dt.normalize()` |
| Data bez godziny, obiekt Python `date` (tylko do eksportu/wyświetlenia) | `.dt.date` |
| Data jako tekst w wybranym formacie | `.dt.strftime("%d.%m.%Y")` |
| Okres (miesiąc/kwartał jako etykieta, nie punkt w czasie) | `.dt.to_period("M")` / `.dt.to_period("Q")` |
| Wartość z poprzedniego wiersza / różnica dzień-do-dnia | `.shift(1)` / `.diff()` |
| Przesunięcie samej daty o kalendarzowy odcinek czasu | `+ pd.DateOffset(months=1)` / `+ pd.Timedelta(days=n)` |
| Grupowanie po miesiącu/tygodniu z kolumny z datą (bez indeksu) | `groupby(pd.Grouper(key="col", freq="ME"))` |
| Agregacja po miesiącu, gdy data jest indeksem | `.resample("ME").sum()` |
| Uzupełnienie brakujących dat bez agregacji (dane już zagregowane, jeden wpis na dzień) | `.asfreq("D", fill_value=...)` |
| Uzupełnienie + agregacja jednocześnie (nieregularne godziny / możliwe duplikaty w indeksie) | `.resample("D").sum()` |
| Uzupełnienie z jawnie zdefiniowanym zakresem dat | `.reindex(pd.date_range(...), fill_value=...)` |

**Uwaga o wersjach pandas:** od pandas 2.2 aliasy częstotliwości `"M"`, `"Q"`, `"Y"` są przestarzałe na rzecz `"ME"`, `"QE"`, `"YE"` (end-of-period). Stary alias nadal działa, ale zgłasza `FutureWarning` — warto od razu używać nowej wersji, żeby nie przepisywać kodu przy kolejnej migracji środowiska.